In [ ]:
print("Jupyter Notebook Working Successfully")

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix,
    roc_auc_score
)

from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier

import pickle
import shap

In [ ]:
df = pd.read_csv("../data/WA_Fn-UseC_-Telco-Customer-Churn.csv")

In [ ]:
df.head()

In [ ]:
df.info()

In [ ]:
df.isnull().sum()

In [ ]:
df.drop("customerID", axis=1, inplace=True)

In [ ]:
df["TotalCharges"] = pd.to_numeric(df["TotalCharges"], errors="coerce")

In [ ]:
df["TotalCharges"] = df["TotalCharges"].fillna(df["TotalCharges"].median())

In [ ]:
df.isnull().sum()

In [ ]:
sns.countplot(x="Churn", data=df)

plt.title("Customer Churn Distribution")

plt.show()

## Insight:
Most customers stayed with the company, but a significant number of customers churned. This indicates the importance of customer retention strategies.

In [ ]:
sns.countplot(x="Contract", hue="Churn", data=df)

plt.title("Contract Type vs Churn")

plt.xticks(rotation=45)

plt.show()

## Insight:
Customers with month-to-month contracts are more likely to leave the company compared to customers with yearly contracts.

In [ ]:
sns.histplot(df["MonthlyCharges"], kde=True)

plt.title("Monthly Charges Distribution")

plt.show()

In [ ]:
sns.boxplot(x="Churn", y="MonthlyCharges", data=df)

plt.title("Monthly Charges vs Churn")

plt.show()

In [ ]:
sns.boxplot(x="Churn", y="tenure", data=df)

plt.title("Tenure vs Churn")

plt.show()

In [ ]:
plt.figure(figsize=(15,10))

sns.heatmap(df.corr(numeric_only=True),
            cmap="coolwarm")

plt.title("Correlation Heatmap")

plt.show()

# EDA Summary

- Customers with high monthly charges tend to churn more.
- Customers with shorter tenure are more likely to leave.
- Month-to-month contracts have higher churn rates.
- Long-term contracts improve customer retention.

In [ ]:
from sklearn.preprocessing import LabelEncoder

# Create encoder
le = LabelEncoder()

# Encode all object columns
for col in df.select_dtypes(include='object').columns:
    
    df[col] = le.fit_transform(df[col])

In [ ]:
df.head()

In [ ]:
X = df.drop("Churn", axis=1)

y = df["Churn"]

In [ ]:
print(X.shape)

print(y.shape)

In [ ]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42
)

In [ ]:
print(X_train.shape)

print(X_test.shape)

In [ ]:
from sklearn.linear_model import LogisticRegression

lr = LogisticRegression(max_iter=5000)

lr.fit(X_train, y_train)

In [ ]:
lr_pred = lr.predict(X_test)

In [ ]:
from sklearn.metrics import accuracy_score

print("Accuracy:", accuracy_score(y_test, lr_pred))

In [ ]:
from sklearn.metrics import classification_report

print(classification_report(y_test, lr_pred))

In [ ]:
from sklearn.metrics import confusion_matrix

cm = confusion_matrix(y_test, lr_pred)

sns.heatmap(cm,
            annot=True,
            fmt='d')

plt.title("Confusion Matrix")

plt.show()

In [ ]:
from sklearn.metrics import roc_auc_score

score = roc_auc_score(y_test, lr_pred)

print("ROC-AUC Score:", score)

In [ ]:
from sklearn.ensemble import RandomForestClassifier

rf = RandomForestClassifier(
    n_estimators=100,
    random_state=42
)

rf.fit(X_train, y_train)

In [ ]:
rf_pred = rf.predict(X_test)

In [ ]:
print("Random Forest Accuracy:")

print(accuracy_score(y_test, rf_pred))

In [ ]:
print(classification_report(y_test, rf_pred))

In [ ]:
cm_rf = confusion_matrix(y_test, rf_pred)

sns.heatmap(cm_rf,
            annot=True,
            fmt='d')

plt.title("Random Forest Confusion Matrix")

plt.show()

In [ ]:
from xgboost import XGBClassifier

xgb = XGBClassifier(
    n_estimators=100,
    learning_rate=0.1,
    max_depth=5,
    random_state=42
)

xgb.fit(X_train, y_train)

In [ ]:
xgb_pred = xgb.predict(X_test)

In [ ]:
print("XGBoost Accuracy:")

print(accuracy_score(y_test, xgb_pred))

In [ ]:
print(classification_report(y_test, xgb_pred))

In [ ]:
cm_xgb = confusion_matrix(y_test, xgb_pred)

sns.heatmap(cm_xgb,
            annot=True,
            fmt='d')

plt.title("XGBoost Confusion Matrix")

plt.show()

In [ ]:
models = pd.DataFrame({
    "Model": [
        "Logistic Regression",
        "Random Forest",
        "XGBoost"
    ],
    
    "Accuracy": [
        accuracy_score(y_test, lr_pred),
        accuracy_score(y_test, rf_pred),
        accuracy_score(y_test, xgb_pred)
    ]
})

models

In [ ]:
sns.barplot(
    x="Model",
    y="Accuracy",
    data=models
)

plt.title("Model Accuracy Comparison")

plt.show()

In [ ]:
import shap

In [ ]:
explainer = shap.Explainer(xgb)

In [ ]:
shap_values = explainer(X_test)

In [ ]:
shap.plots.beeswarm(shap_values)

In [ ]:
shap.plots.bar(shap_values)

# SHAP Insights

- Customers with high monthly charges are more likely to churn.
- Customers with short tenure have higher churn probability.
- Long-term contracts reduce customer churn.
- Internet service type strongly affects customer retention.

In [ ]:
import pickle

In [ ]:
pickle.dump(xgb, open("../models/churn_model.pkl", "wb"))